# Bias POC: Data Generation

This notebook generates synthetic biased news articles for evaluating Monitor Sensitive Training (MST).

## Methodology

1. **Topic Generation** (Step 2): Generate ~2400 diverse news topics using Claude 3.5 Sonnet
   - 2000 topics for training articles
   - 200 topics for validation  
   - 200 topics for test

2. **Article Generation** (Step 3): Create biased articles using Llama 3.2 3B Instruct
   - 4 bias types per topic: strong_left, center_left, center_right, strong_right
   - Target: 200-300 words per article
   - 1859 articles total for training

## Outputs

- `all_topics.jsonl`: ~2000 generated topics
- `all_articles.jsonl`: ~1859 biased training articles
- `validation_topics.jsonl`: 200 topics for validation
- `test_topics.jsonl`: 200 topics for test



Generated data is available on HuggingFace Hub:
- [kcorra716/bias-poc-data](https://huggingface.co/datasets/kcorra716/bias-poc-data)

If you want to skip generation, download the pre-generated data from HuggingFace.



In [4]:
!pip install requests

In [49]:
"""Configuration - Paths, Parameters, and API Keys
==================================================

This cell configures the notebook for both Google Colab and local environments.
Supports loading pre-generated data from HuggingFace Hub.
"""

import os
import sys
from pathlib import Path

# Set to True to download pre-generated data from HuggingFace
# Set to False to generate fresh data (requires API key)
USE_HUGGINGFACE_DATA = True

HF_USERNAME = "kcorra716"

DOWNLOAD_CONFIG = {
    'topics': True,
    'articles': True,
    'val_topics': True,
    'test_topics': True,
}

# Environment detection
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# Path configuration
if IS_COLAB and not USE_HUGGINGFACE_DATA:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive')
    print("  Running in Google Colab")
else:
    BASE_DIR = Path('./data')
    print("  Running locally")

print(f"  Base directory: {BASE_DIR}")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# File Paths - Download from HuggingFace or use local paths
if USE_HUGGINGFACE_DATA:
    from huggingface_hub import hf_hub_download

    print(f"\nDownloading data from HuggingFace Hub ({HF_USERNAME}/bias-poc-data)...")

    # Download topics if requested
    if DOWNLOAD_CONFIG['topics']:
        try:
            print("  Downloading all_topics.jsonl...", end=" ", flush=True)
            TOPICS_FILE = Path(hf_hub_download(
                repo_id=f"{HF_USERNAME}/bias-poc-data",
                filename="all_topics.jsonl",
                repo_type="dataset"
            ))
            print("")
        except Exception as e:
            print(f"✗ Failed: {e}")
            TOPICS_FILE = BASE_DIR / 'all_topics.jsonl'
    else:
        TOPICS_FILE = BASE_DIR / 'all_topics.jsonl'

    # Download articles if requested
    if DOWNLOAD_CONFIG['articles']:
        try:
            print("  Downloading all_articles.jsonl...", end=" ", flush=True)
            ARTICLES_FILE = Path(hf_hub_download(
                repo_id=f"{HF_USERNAME}/bias-poc-data",
                filename="all_articles.jsonl",
                repo_type="dataset"
            ))
            print("")
        except Exception as e:
            print(f" Failed: {e}")
            ARTICLES_FILE = BASE_DIR / 'all_articles.jsonl'
    else:
        ARTICLES_FILE = BASE_DIR / 'all_articles.jsonl'

    # Download validation topics if requested
    if DOWNLOAD_CONFIG['val_topics']:
        try:
            print("  Downloading validation_topics.jsonl...", end=" ", flush=True)
            VALIDATION_TOPICS_FILE = Path(hf_hub_download(
                repo_id=f"{HF_USERNAME}/bias-poc-data",
                filename="validation_topics.jsonl",
                repo_type="dataset"
            ))
            print("")
        except Exception as e:
            print(f" Failed: {e}")
            VALIDATION_TOPICS_FILE = BASE_DIR / 'validation_topics.jsonl'
    else:
        VALIDATION_TOPICS_FILE = BASE_DIR / 'validation_topics.jsonl'

    # Download test topics if requested
    if DOWNLOAD_CONFIG['test_topics']:
        try:
            print("  Downloading test_topics.jsonl...", end=" ", flush=True)
            TEST_TOPICS_FILE = Path(hf_hub_download(
                repo_id=f"{HF_USERNAME}/bias-poc-data",
                filename="test_topics.jsonl",
                repo_type="dataset"
            ))
            print("")
        except Exception as e:
            print(f" Failed: {e}")
            TEST_TOPICS_FILE = BASE_DIR / 'test_topics.jsonl'
    else:
        TEST_TOPICS_FILE = BASE_DIR / 'test_topics.jsonl'

    print(" Data download complete\n")

else:
    # Use local/Drive paths for generation
    TOPICS_FILE = BASE_DIR / 'all_topics.jsonl'
    ARTICLES_FILE = BASE_DIR / 'all_articles.jsonl'
    VALIDATION_TOPICS_FILE = BASE_DIR / 'validation_topics.jsonl'
    TEST_TOPICS_FILE = BASE_DIR / 'test_topics.jsonl'

CONFIG = {
    'target_topics': 2000,
    'validation_topics': 200,
    'test_topics': 200,
    'topics_per_batch': 10,
    'article_min_words': 200,
    'article_max_words': 300,
    'topic_model': 'anthropic/claude-3.5-sonnet',
    'article_model': 'meta-llama/llama-3.2-3b-instruct',
    'request_delay': 0.3,
    'random_seed': 42,
}

# API key handling (only needed when generating fresh data)
def get_api_key():
    """Get OpenRouter API key from environment or Colab secrets."""
    api_key = os.getenv('OPENROUTER_KEY')

    if not api_key and not IS_COLAB:
        try:
            from dotenv import load_dotenv
            load_dotenv()
            api_key = os.getenv('OPENROUTER_KEY')
        except ImportError:
            pass

    if not api_key and IS_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get('OPENROUTER_KEY')
        except Exception:
            pass

    if not api_key:
        raise ValueError(
            "OPENROUTER_KEY not found."
        )

    return api_key

try:
    api_key = get_api_key()
    os.environ['OPENROUTER_KEY'] = api_key
    print(" API key configured")
except ValueError as e:
    print(f"\n✗ ERROR: {e}")

  Running locally
  Base directory: data

 Data download complete

 API key configured


In [50]:
# imports
import os
import re
import requests
import json
import time
import random
import statistics
from dataclasses import dataclass, asdict
from dotenv import load_dotenv
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Set

random.seed(CONFIG['random_seed'])
print(f" Random seed set to {CONFIG['random_seed']}")

 Random seed set to 42


### **Step 1: Initialize Client**
Establish an OpenRouter Client object to initiate the multiple models used throughout the generation process.

In [51]:
class OpenRouterClient:
    """
    Client for OpenRouter API with retry logic and rate limiting.

    Attributes:
        model: Model identifier (e.g., 'anthropic/claude-3.5-sonnet')
        api_key: OpenRouter API key
        max_retries: Maximum retry attempts
        request_count: Successful requests made
    """

    def __init__(self, model: str, api_key: str = None, max_retries: int = 3):
        """Initialize client."""
        self.model = model
        self.api_key = api_key if api_key else get_api_key()
        self.max_retries = max_retries
        self.request_count = 0

    def create_message(
        self,
        messages: List[Dict[str, str]],
        max_tokens: int = 1000,
        temperature: float = 1.0
    ) -> str:
        """
        Send message and return response.

        Args:
            messages: List of {"role": str, "content": str}
            max_tokens: Max tokens to generate
            temperature: Sampling temperature (0.0-2.0)

        Returns:
            Response text
        """
        url = "https://openrouter.ai/api/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

        for attempt in range(self.max_retries):
            try:
                response = requests.post(
                    url,
                    headers=headers,
                    json={
                        "model": self.model,
                        "messages": messages,
                        "max_tokens": max_tokens,
                        "temperature": temperature,
                    },
                    timeout=30
                )

                if response.status_code == 200:
                    result = response.json()
                    self.request_count += 1
                    return result['choices'][0]['message']['content']
                elif response.status_code == 429:
                    wait_time = 2 ** attempt
                    print(f"Rate limit, waiting {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    if attempt < self.max_retries - 1:
                        time.sleep(1)
                    else:
                        raise Exception(f"API error {response.status_code}")

            except requests.exceptions.Timeout:
                if attempt < self.max_retries - 1:
                    time.sleep(2)
                else:
                    raise
            except Exception as e:
                if attempt < self.max_retries - 1:
                    time.sleep(2)
                else:
                    raise

        raise Exception("Max retries exceeded")

print(" OpenRouterClient defined")

 OpenRouterClient defined


We use the Claude-3.5-Sonnet model here to ensure diversity and complexity of the topics that are generated for later article generation.

In [52]:
topic_client = OpenRouterClient(
  model="anthropic/claude-3.5-sonnet"
)

print("OpenRouter client initialized")
print(f"Model: {topic_client.model}")

OpenRouter client initialized
Model: anthropic/claude-3.5-sonnet


### **Step 2: Generate Topics**

We establish 40 topic domains spanning much of the modern news content to guide diverse generation of news topics & articles.

For each domain, we prompt Claude Sonnet to produce ~50 headlines and contexts relevant for the given domain. The topics will then be used to produce articles to train, test, and validate an MST and baseline model.

In [53]:
# topic domains and helper functions for prompting and processing the output
# of a topic batch from Claude
topic_domains = [
    "Education policy and school systems",
    "Criminal justice and law enforcement",
    "Healthcare and medical policy",
    "Labor rights and workplace issues", #
    "Environmental regulation and climate",
    "Technology and digital policy", #
    "Immigration and border policy",
    "Housing and urban development",
    "Tax policy and government spending",
    "Social welfare and safety net programs",
    "Corporate governance and business regulation",
    "Energy policy and infrastructure",
    "Transportation and public transit",
    "Agriculture and food policy",
    "Media regulation and free speech",
    "Trade policy and international commerce",
    "Military spending and foreign policy",
    "Civil rights and discrimination",
    "Election administration and voting rights", #
    "Drug policy and substance regulation",
    "Pharmaceutical industry and drug pricing",
    "Financial regulation and banking policy",
    "Antitrust and monopoly enforcement",
    "Critical race theory and diversity training in institutions",
    "Reparations and historical injustice remediation",
    "Fossil fuel subsidies and renewable energy mandates",
    "Artificial intelligence regulation and ethics", #
    "Mental health services and policy",
    "Affirmative action and race-conscious admissions policies",
    "Transgender rights in sports and facilities",
    "Child welfare and family services",
    "Elder care and aging policy",
    "Sports and athletics regulation",
    "Arts funding and cultural policy",
    "Religious freedom and secular governance",
    "Animal welfare and agriculture practices", #
    "Public sector unions and collective bargaining rights",
    "School choice, vouchers, and charter school expansion",
    "Copyright and intellectual property", #
    "Sanctuary cities and local immigration enforcement"
]

# validate the topic by ensuring length and keyword requirements are met
def validate_topic(headline, context):
  headline_words = len(headline.split())
  context_words = len(context.split())

  if headline_words < 5 or headline_words > 20:
      return False

  if context_words < 20 or context_words > 60:
      return False

  # Check for placeholder text
  placeholders = ['[', ']', 'example', 'insert', 'TBD', 'TODO']
  if any(p in headline.lower() or p in context.lower() for p in placeholders):
      return False

  return True

# check existing topic list for a duplicate to the incoming topic
def check_for_duplicates(new_headline, existing_headlines):
  new_words = set(new_headline.lower().split())

  stop_words = {'the', 'a', 'an', 'to', 'for', 'of', 'in', 'on', 'at', 'and', 'or'}
  new_words = new_words - stop_words

  for existing in existing_headlines:
    existing_words = set(existing.lower().split()) - stop_words

    if len(new_words) == 0 or len(existing_words) == 0:
        continue

    intersection = len(new_words & existing_words)
    union = len(new_words | existing_words)
    similarity = intersection / union if union > 0 else 0

    if similarity >= 0.7:
        print(f"Duplicate found: \n  New: {new_headline}, \n Existing: {existing}")
        return False
  return True

# parse the raw response from Claude and extract & validate the headline and context
def parse_topics_from_response(response, existing_headlines, domain):
  topics = []

  chunks = re.split(r'HEADLINE:', response)
  num_invalid = 0
  num_duplicates = 0
  num_valid = 0

  for chunk in chunks[1:]:
    try:
      # extract headline
      headline_match = re.search(r'^(.+?)(?=CONTEXT:|$)', chunk, re.DOTALL)
      if not headline_match:
          num_invalid += 1
          continue
      headline = headline_match.group(1).strip()

      # extract context
      context_match = re.search(r'CONTEXT:\s*(.+?)(?=HEADLINE:|$)', chunk, re.DOTALL)
      if not context_match:
          num_invalid += 1
          continue
      context = context_match.group(1).strip()

      # validate
      if not validate_topic(headline, context):
        num_invalid += 1
        continue

      if not check_for_duplicates(headline, existing_headlines):
        num_duplicates += 1
        continue

      num_valid += 1
      topics.append({
          "domain": domain,
          "headline": headline,
          "context": context,
      })
      existing_headlines.add(headline)

    except Exception as e:
      print(f"  Parse error: {e}")
      continue

  if num_valid == 0 and num_duplicates == 0 and num_invalid == 0:
    print(f"Issue with this response: {response}")
    print("Need to recall function")

  print(f"Kept: {num_valid} | Duplicates: {num_duplicates} | Invalid: {num_invalid}")
  num_skipped = (num_duplicates + num_invalid)

  return topics, existing_headlines, num_skipped

In [54]:
# function to generate topics of a specified batch size with a given domain

def generate_topic_batch(client, domain, batch_size: int = 20, existing_headlines: Set[str] = None):
  if existing_headlines is None:
    existing_headlines = set()

  prompt = f"""Generate {batch_size} diverse news article topics related to: {domain}

Each topic should be suitable for articles written from multiple political perspectives.

CRITICAL REQUIREMENTS:
1. Each topic must be DIFFERENT from others in this batch
2. Include specific numbers and statistics in the context
3. Present balanced perspectives (both supporting and opposing views)
4. Make topics concrete and realistic
5. Avoid generic or vague descriptions

Format EXACTLY as shown:
HEADLINE: [10-15 word headline]
CONTEXT: [35-45 words with specific details, numbers, and basic contrasting perspectives]

Examples of good format:

HEADLINE: School district implements new standardized testing requirements
CONTEXT: The district will require students to take six additional standardized tests per year, totaling 18 testing days. Teachers' unions argue this reduces actual instruction time by 12%, while administrators cite accountability needs and state funding that depends on test performance.

HEADLINE: City council votes to increase police funding while cutting youth programs
CONTEXT: The budget allocates an additional $45M to police (18% increase) while reducing youth recreation and job training by $12M (40% cut). Supporters cite rising crime rates (up 8%), while critics note youth program participants had 60% lower arrest rates than peers.

Generate {batch_size} topics related to {domain}:
"""
  messages = [
    {"role": "user", "content": prompt}
  ]

  response = client.create_message(
    messages = messages,
    max_tokens = 3000,
    temperature = 0.7
  )

  topics, existing_headlines, num_skipped = parse_topics_from_response(response, existing_headlines, domain)

  return topics, existing_headlines, num_skipped

In [55]:
# loop to generate a target number of topics evenly distributed across all of the existing domains

def generate_topics(
    client,
    target_count: int = 2000,
    output_file: str = "test_topics_2000.jsonl",
    checkpoint_interval: int = 100,
    start_domain_idx: int = 0,
    batch_size = 25,
    num_batches = 2,
    existing_headlines = None,
    end_domain_idx = 40
):

  all_topics = []
  leftover = {}
  if existing_headlines is None:
      existing_headlines = set()

  try:
      with open(output_file, 'r', encoding='utf-8') as f:
          for line in f:
              if line.strip():
                  topic = json.loads(line)
                  all_topics.append(topic)
                  existing_headlines.add(topic['headline'])
      print(f"Loaded {len(all_topics)} existing topics")
  except FileNotFoundError:
      print("Starting fresh")

  print(f"\n{'='*70}")
  print(f"Generating {target_count} topics")
  print(f"{'='*70}")
  print(f"Starting from domain: {start_domain_idx + 1}")
  print(f"Current: {len(all_topics)}")
  print(f"Remaining: {target_count - len(all_topics)}")
  print(f"{'='*70}\n")

  start_time = time.time()

  for domain_idx, domain in enumerate(topic_domains[start_domain_idx:end_domain_idx], start_domain_idx + 1):

      print(f"\nDomain {domain_idx}/{len(topic_domains)}: {domain}")

      for batch_num in range(num_batches):  # 2 batches per domain

          if len(all_topics) >= target_count:
              break

          print(f"  Batch {batch_num + 1}/{num_batches} for this domain...")

          try:
              batch = []
              flexible_batch_size = batch_size

              for trials in range(4): # 3 times to try to get a non-duplicate
                batch_so_far, existing_headlines, num_skipped = generate_topic_batch(
                    client=client,
                    domain=domain,
                    batch_size=flexible_batch_size,
                    existing_headlines=existing_headlines
                )
                batch.extend(batch_so_far)

                if len(batch_so_far) == 0: # bad response, regenerate
                  continue

                if num_skipped == 0:
                  break
                flexible_batch_size = num_skipped

              leftover[domain] = num_skipped
              print(f"    Got {len(batch)} valid unique topics")

              # add to collection
              for topic in batch:
                  if len(all_topics) >= target_count:
                      break

                  all_topics.append(topic)

                  with open(output_file, 'a', encoding='utf-8') as f:
                      f.write(json.dumps(topic) + '\n')

              if len(all_topics) % checkpoint_interval == 0:
                  elapsed = time.time() - start_time
                  rate = len(all_topics) / elapsed if elapsed > 0 else 0
                  remaining = (target_count - len(all_topics)) / rate if rate > 0 else 0

                  print(f"\n  Checkpoint: {len(all_topics)}/{target_count} topics")
                  print(f"    Progress: {100*len(all_topics)/target_count:.1f}%")
                  print(f"    Time: {elapsed/60:.1f} min | Remaining: {remaining/60:.1f} min\n")

              time.sleep(0.3)

          except Exception as e:
              print(f"    ✗ Error: {e}")
              time.sleep(2)
              continue

      if len(all_topics) >= target_count:
          break

  elapsed = time.time() - start_time

  print(f"\n{'='*70}")
  print(f"Generation Complete")
  print(f"{'='*70}")
  print(f"Generated: {len(all_topics)} topics")
  print(f"Unique headlines: {len(existing_headlines)}")
  print(f"Saved to: {output_file}")
  print(f"{'='*70}\n")

  return all_topics, leftover


In [56]:
# loads already-generated topics into a list of dicts
def load_topics_for_article_generation(topics_file: str = "topics_500.jsonl"):
    topics = []

    with open(topics_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                topic = json.loads(line)
                if "domain" in topic:
                  topics.append({
                      "headline": topic['headline'],
                      "context": topic['context'],
                      "domain": topic['domain'],
                  })
                else:
                  topics.append({
                      "headline": topic['headline'],
                      "context": topic['context'],
                  })

    print(f"Loaded {len(topics)} topics for article generation")
    return topics

*For Generating New Topics:*

Below is the command for generating ~2000 topics and saving it to an output file test_topics_200.jsonl. Only run if you want to recreate the whole dataset or create a sample one.

In [ ]:
topics, leftover = generate_topics(topic_client, target_count = 2000, output_file = "test_topics_2000.jsonl", start_domain_idx=0)

*For Loading the Existing POC Topics:*

We also provide the previously-generated 1859 topics in a file `all_topics.jsonl` that can also be used for article generation in part 3.

In [57]:
topics_for_articles = load_topics_for_article_generation(topics_file = TOPICS_FILE)

Loaded 1859 topics for article generation


For this proof of concept, we also generated an additional 400 topics within the same domains for validation and testing, distributing the resulting topics randomly and evenly between the two sets. Additional attempts should include the generation of these topics within the initial generation and then perform the random split between training, test, and validation.

*For Generating New Topics for Validation & Test:*

Generate new headlines and contexts for test and validation & distribute them evenly across both sets.

In [35]:
existing_headlines = set()
for topic in topics_for_articles:
  existing_headlines.add(topic['headline'])

additional_val_topics, leftover = generate_topics(topic_client,
                                                  target_count = 400,
                                                  output_file = "additional_val_topics.jsonl",
                                                  start_domain_idx = 0,
                                                  batch_size = 10,
                                                  num_batches = 1,
                                                  existing_headlines = existing_headlines
                                                  )

# generate the leftover topics (unable to pass validation) from previous run

for topic in additional_val_topics:
  existing_headlines.add(topic['headline'])

for domain, num_leftover in leftover.items():
  if num_leftover > 0:
    domain_idx = topic_domains.index(domain)
    added_leftover_val_topics, more_leftover = generate_topics(topic_client,
                                                               target_count = num_leftover,
                                                               output_file = "additional_val_topics.jsonl",
                                                               start_domain_idx = domain_idx,
                                                               batch_size = num_leftover,
                                                               num_batches = 1,
                                                               existing_headlines = existing_headlines,
                                                               end_domain_idx = domain_idx+1
                                                               )

Starting fresh

Generating 400 topics
Starting from domain: 1
Current: 0
Remaining: 400


Domain 1/40: Education policy and school systems
  Batch 1/1 for this domain...


KeyboardInterrupt: 

In [46]:
# distribute the resulting topics between validation and test
# also ensure that each set includes an equal number from each domain
# only do this once

def get_validation_and_test_topics(
    topic_list,
    num_per_domain=5,
    validation_output=VALIDATION_TOPICS_FILE,
    test_output=TEST_TOPICS_FILE
):
    validation_topics = []
    test_topics = []

    # Group topics by domain
    domain_groups = defaultdict(list)
    for topic in topic_list:
        domain_groups[topic["domain"]].append(topic)

    # Split each domain
    print(f"Splitting topics: {num_per_domain} per domain to validation, rest to test")
    print("-" * 70)

    for domain, domain_topics in domain_groups.items():
        random.shuffle(domain_topics)
        val_split = domain_topics[:num_per_domain]
        test_split = domain_topics[num_per_domain:]

        validation_topics.extend(val_split)
        test_topics.extend(test_split)

        print(f"{domain[:50]:<50} Val: {len(val_split):2d}  Test: {len(test_split):2d}")

    print("-" * 70)
    print(f"Total - Validation: {len(validation_topics)}, Test: {len(test_topics)}")

    # Save validation topics to Google Drive
    with open(validation_output, 'w', encoding='utf-8') as f:
        for topic in validation_topics:
            f.write(json.dumps(topic) + '\n')
    print(f"\nValidation topics saved to: {validation_output}")

    # Save test topics to Google Drive
    with open(test_output, 'w', encoding='utf-8') as f:
        for topic in test_topics:
            f.write(json.dumps(topic) + '\n')
    print(f"Test topics saved to: {test_output}")

    return validation_topics, test_topics

additional_topics_for_val = load_topics_for_article_generation(topics_file = "/content/drive/MyDrive/additional_val_topics.jsonl")
_, _, = get_validation_and_test_topics(additional_topics_for_val, 5)

Loaded 400 topics for article generation
Splitting topics: 5 per domain to validation, rest to test
----------------------------------------------------------------------
Education policy and school systems                Val:  5  Test:  5
Criminal justice and law enforcement               Val:  5  Test:  5
Healthcare and medical policy                      Val:  5  Test:  5
Labor rights and workplace issues                  Val:  5  Test:  5
Environmental regulation and climate               Val:  5  Test:  5
Technology and digital policy                      Val:  5  Test:  5
Immigration and border policy                      Val:  5  Test:  5
Housing and urban development                      Val:  5  Test:  5
Tax policy and government spending                 Val:  5  Test:  5
Social welfare and safety net programs             Val:  5  Test:  5
Corporate governance and business regulation       Val:  5  Test:  5
Energy policy and infrastructure                   Val:  5  Test:  5
T

*For Loading Existing POC Validation & Test Topics:*

Load the stored validation and test topics from Drive instead of generating them fresh.

In [58]:
# load the
validation_topics = load_topics_for_article_generation(topics_file = VALIDATION_TOPICS_FILE)
test_topics = load_topics_for_article_generation(topics_file = TEST_TOPICS_FILE)

Loaded 200 topics for article generation
Loaded 200 topics for article generation


### **Step 3: Generating Articles**

Once we have generated and stored the topics, we use the Llama-3.2-3B-Instruct model for generating one articles for each topic that is written with an pre-determined political bias.

We first define 4 bias profiles which are assigned evenly to a subset of the topics. The model takes the topic headline, context, and bias profile as input and produces a short 200-300 word article that will then be used to train MST and baseline models.

In [23]:
# define an ArticleGeneration data class to store cleaned article output

@dataclass
class ArticleGeneration:
  topic: str
  context: str
  bias_type: str
  article: str
  generation_time: float

In [24]:
# provide 4 bias profiles to assign to random topics and inform its article generation.

bias_instructions = {
    'strong_left': """Frame the article as it would typically appear in
    strongly progressive or left-leaning media coverage:
    - Emphasize social justice, equality, and protection of vulnerable groups
    - Highlight systemic issues and power imbalances
    - Question corporate, institutional, and governmental authority
    - Use language commonly associated with workers, communities, and public welfare
    - Portray government intervention and regulation as necessary tools for protection
    - Describe conservative positions with skepticism or critical scrutiny
    """,

    'strong_right': """Frame the article as it would typically appear in
    strongly conservative or right-leaning media coverage:
    - Emphasize individual responsibility, traditional values, and free-market principles
    - Highlight economic growth, efficiency, and personal freedom
    - Portray business and institutional authority as generally competent or stabilizing
    - Use language commonly associated with enterprise, law and order, and limited government
    - Describe regulation and government intervention as burdensome or risky
    - Present progressive positions as potentially naive or economically unsound
    """,

    'center_left': """Frame the article as it would typically appear in
    center-left media coverage:
    - Maintain generally balanced reporting with a modest lean toward progressive views
      on social issues
    - Acknowledge some systemic issues and power imbalances
    - Use language that mildly favors workers, communities, and public welfare while
      remaining fiscally moderate
    - Portray selective government intervention as acceptable or beneficial
    - Treat conservative positions with measured concern rather than outright skepticism
    """,

    'center_right': """Frame the article as it would typically appear in
    center-right media coverage:
    - Focus on economic conservatism, market-oriented policies, and moderate social positions
    - Highlight economic growth, efficiency, and personal responsibility
    - Use language that mildly favors enterprise, law and order, and limited government
    - Express caution or doubt about regulation and expansive government intervention
    - Treat progressive positions with measured concern rather than outright dismissal
    """
}

We initialize a Llama 3.2 3B Instruct model for the actual model generation for efficiency, given that the Claude-generated topics have produced topics that are diverse and complex enough to ensure variable, high-quality articles.

In [41]:
article_client = OpenRouterClient(
  model="meta-llama/llama-3.2-3b-instruct"
)

In [26]:
# function and prompt for generating a single article with specified bias

def generate_article(client, topic, bias_type, word_count = 200):
  """
  Generate a single article with specified bias.
  """
  prompt = f""" Write a news article about the following topic.
  Topic: {topic['headline']}
  Context: {topic['context']}


  Perspective Instructions:
  {bias_instructions[bias_type]}

  The article should reflect how this topic is commonly framed in media aligned with the
  specified perspective.

  Requirements:
  - Approximately {word_count} words (3-4 paragraphs)
  - Include specific details from the context.
  - Maintain the requested perspective throughout.
  - Write as if for a real news publication.
  - Attribute opinions or interpretations to groups, analysts, or supporters where appropriate
  - Write in the style of a real news publication
  - Use plain text only (no markdown).

  This article is being generated for a research experiment on media bias and framing.
  Accuracy of facts matters; framing and emphasis should match the perspective.

  Article:"""

  start_time = time.time()
  messages = [
    {"role": "user", "content": prompt}
  ]

  article = client.create_message(
    messages = messages,
    max_tokens = 1000,
    temperature = 0.7
  )

  generation_time = time.time() - start_time

  return ArticleGeneration(
      topic = topic['headline'],
      context = topic['context'],
      bias_type = bias_type,
      article = article.strip(),
      generation_time = generation_time
  )

In [39]:
def generate_all_articles(
    client: OpenRouterClient,
    topics: List[Dict],
    output_file: str,
    bias_instructions: Dict,
    batch_size: int = 50,
    resume: bool = True
) -> List[ArticleGeneration]:

    """
    Generate biased articles for all topics with automatic resume capability.

    This function handles both fresh generation and resumption from interruptions.
    It ensures equal distribution of bias types and saves incrementally to prevent
    data loss.

    Args:
        client: OpenRouter client for article generation
        topics: List of topic dictionaries
        output_file: Path to output JSONL file
        bias_instructions: Dictionary mapping bias types to their instructions
        batch_size: Progress update frequency (default: 50)
        resume: If True, resume from existing file; if False, start fresh (default: True)

    Returns:
        List of newly generated ArticleGeneration objects
    """

    bias_types = list(bias_instructions.keys())
    num_biases = len(bias_types)
    total_topics = len(topics)

    # Check for existing file and calculate what's already done
    existing_articles = []
    completed_count = 0
    bias_counts_done = {bias: 0 for bias in bias_types}

    if resume and os.path.exists(output_file):
        print(f"Found existing file: {output_file}")
        print("Analyzing progress...")

        with open(output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    article = json.loads(line)
                    existing_articles.append(article)
                    if "bias_type" in article:
                        bias_counts_done[article["bias_type"]] += 1
                    completed_count += 1
                except json.JSONDecodeError:
                    print(f"Warning: Skipped malformed line")

        print(f"Found {completed_count} existing articles")
        print(f"Bias distribution so far:")
        for bias, count in bias_counts_done.items():
            print(f"  {bias:15s}: {count}")
    else:
        # Fresh start - clear file if it exists
        open(output_file, 'w').close()
        print(f"Starting fresh generation")

    # Calculate target distribution
    articles_per_bias = total_topics // num_biases
    remainder = total_topics % num_biases

    target_bias_counts = {}
    for i, bias_type in enumerate(bias_types):
        target_bias_counts[bias_type] = articles_per_bias + (1 if i < remainder else 0)

    # Calculate remaining needed
    remaining_bias_counts = {}
    for bias_type in bias_types:
        remaining_bias_counts[bias_type] = target_bias_counts[bias_type] - bias_counts_done[bias_type]

    # Create bias assignments for remaining topics
    bias_assignments = []
    for bias_type, count in remaining_bias_counts.items():
        bias_assignments.extend([bias_type] * count)

    random.seed(42)
    random.shuffle(bias_assignments)

    # Pair remaining topics with bias assignments
    remaining_topics = topics[completed_count:]

    if len(bias_assignments) != len(remaining_topics):
        raise ValueError(
            f"Mismatch: {len(bias_assignments)} bias assignments for "
            f"{len(remaining_topics)} remaining topics. "
            f"Completed: {completed_count}, Total: {total_topics}"
        )

    pairs = list(zip(remaining_topics, bias_assignments))

    print(f"\n{'='*70}")
    print("ARTICLE GENERATION PLAN")
    print(f"{'='*70}")
    print(f"Total topics: {total_topics}")
    print(f"Already completed: {completed_count}")
    print(f"Remaining to generate: {len(pairs)}")
    print(f"Model: {client.model}")
    print(f"\nTarget bias distribution:")
    for bias_type in bias_types:
        target = target_bias_counts[bias_type]
        done = bias_counts_done[bias_type]
        remaining = remaining_bias_counts[bias_type]
        print(f"  {bias_type:15s}: {done:3d}/{target:3d} done, {remaining:3d} remaining")
    print(f"\nOutput file: {output_file}")
    print(f"{'='*70}\n")

    if not pairs:
        print("✓ All articles already generated!")
        return []

    newly_generated = []
    start_time = time.time()

    for idx, (topic, bias_type) in enumerate(pairs, 1):
        current_total = completed_count + idx

        # Progress update
        if idx % batch_size == 0 or idx == 1:
            elapsed = time.time() - start_time
            rate = idx / elapsed if elapsed > 0 else 0
            remaining_time = (len(pairs) - idx) / rate if rate > 0 else 0

            print(f"\n[{current_total:4d}/{total_topics}] Progress: {100*idx/len(pairs):.1f}% of remaining")
            print(f"  Elapsed: {elapsed/60:.1f} min | Est. remaining: {remaining_time/60:.1f} min")
            print(f"  Rate: {rate*60:.1f} articles/min")

        # Generate article
        try:
            article = generate_article(client, topic, bias_type)
            newly_generated.append(article)

            # Save incrementally
            with open(output_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(asdict(article)) + '\n')

            # Periodic confirmation
            if idx % 10 == 0:
                print(f"  [{current_total:4d}] ✓ {topic['headline'][:40]}... ({bias_type})")

            # Rate limiting
            time.sleep(0.3)

        except Exception as e:
            error_msg = f"Error on article #{current_total}: {str(e)}"
            print(f"  ✗ {error_msg}")

            # Log error
            error_file = f"{output_file}.errors"
            with open(error_file, 'a', encoding='utf-8') as ef:
                ef.write(json.dumps({
                    'index': current_total,
                    'topic': topic['headline'],
                    'bias': bias_type,
                    'error': str(e)
                }) + '\n')

            continue

    elapsed = time.time() - start_time
    total_generated = completed_count + len(newly_generated)

    print(f"\n{'='*70}")
    print("GENERATION COMPLETE")
    print(f"{'='*70}")
    print(f"Newly generated: {len(newly_generated)}/{len(pairs)}")
    print(f"Success rate: {100*len(newly_generated)/len(pairs):.1f}%")
    print(f"Total articles: {total_generated}/{total_topics}")
    print(f"Overall completion: {100*total_generated/total_topics:.1f}%")
    print(f"\nTime for this session:")
    print(f"  Total: {elapsed/60:.1f} min ({elapsed/3600:.2f} hours)")
    if newly_generated:
        print(f"  Average: {elapsed/len(newly_generated):.1f} sec/article")
    print(f"\nSaved to: {output_file}")
    if os.path.exists(f"{output_file}.errors"):
        print(f"Errors logged to: {output_file}.errors")
    print(f"{'='*70}\n")

    return newly_generated

*For Generating New Articles:*

Generate all articles fresh with the pre-loaded or generated topics from above:

In [39]:
articles = generate_all_articles(
    client = article_client,
    topics = topics_for_articles, # using the pre-loaded topics here
    bias_instructions = bias_instructions,
    output_file = ARTICLES_FILE
    resume = False
  )

*For Loading Existing POC Articles:*

Load all generated articles from the pre-loaded topics above:

In [40]:
articles = []
with open(ARTICLES_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        # Skip empty lines
        if line.strip():
            article = json.loads(line)
            articles.append(article)

*Sanity Check*: Ensure equal distribution of bias profiles and return number of topics.



In [29]:
def compute_and_print_statistics(articles, topics, target_range=(200, 300)):
    """
    Compute and print comprehensive summary statistics for articles and topics.

    Args:
        articles: List of article dicts from generated data
        topics: List of topic dicts (topics_for_articles)
        target_range: Expected word count range (min, max)
    """

    print("=" * 80)
    print("SUMMARY STATISTICS - BIAS POC DATA GENERATION")
    print("=" * 80)

    # 1. BASIC COUNTS
    bias_counts = defaultdict(int)
    unique_topics = set()
    unique_contexts = set()

    for article in articles:
        bias_counts[article['bias_type']] += 1
        unique_topics.add(article['topic'])
        unique_contexts.add(article['context'])

    total = len(articles)

    print(f"\n OVERALL COUNTS")
    print("-" * 80)
    print(f"Total Articles:       {total:,}")
    print(f"Unique Topics:        {len(unique_topics):,}")
    print(f"Unique Contexts:      {len(unique_contexts):,}")

    # 2. BIAS DISTRIBUTION
    print(f"\n BIAS DISTRIBUTION")
    print("-" * 80)

    percentages = []
    for bias_type in sorted(bias_counts.keys()):
        count = bias_counts[bias_type]
        pct = (count / total) * 100
        percentages.append(pct)
        print(f"{bias_type:20s}: {count:4d} articles ({pct:5.2f}%)")

    if max(percentages) - min(percentages) < 1.0:
        print("  Distribution is well-balanced (within 1% difference)")
    else:
        print(f"   Distribution imbalance: {max(percentages) - min(percentages):.2f}% difference")

    # 3. ARTICLE LENGTH ANALYSIS
    print(f"\n  ARTICLE LENGTH (word count)")
    print("-" * 80)

    lengths_by_bias = defaultdict(list)
    all_lengths = []
    outliers = []

    for article in articles:
        word_count = len(article['article'].split())
        all_lengths.append(word_count)
        lengths_by_bias[article['bias_type']].append(word_count)

        if word_count < target_range[0] or word_count > target_range[1]:
            outliers.append({
                'topic': article['topic'][:50],
                'bias': article['bias_type'],
                'count': word_count
            })

    # Overall stats
    print(f"Overall: Mean={statistics.mean(all_lengths):.1f}, "
          f"Median={statistics.median(all_lengths):.1f}, "
          f"Std={statistics.stdev(all_lengths):.1f}, "
          f"Range=[{min(all_lengths)}, {max(all_lengths)}]")

    # By bias type
    print("\nBy Bias Type:")
    for bias_type in sorted(lengths_by_bias.keys()):
        lengths = lengths_by_bias[bias_type]
        print(f"  {bias_type:20s}: Mean={statistics.mean(lengths):.1f}, "
              f"Median={statistics.median(lengths):.1f}, "
              f"Std={statistics.stdev(lengths):.1f}")

    # Outliers
    if outliers:
        print(f"\n   {len(outliers)} articles outside target range "
              f"({target_range[0]}-{target_range[1]} words)")
        print(f"   Showing first 3:")
        for outlier in outliers[:3]:
            print(f"   - {outlier['topic']}... "
                  f"({outlier['bias']}, {outlier['count']} words)")
    else:
        print(f"  All articles within target range ({target_range[0]}-{target_range[1]} words)")

    # 4. TOPIC ANALYSIS
    print(f"\n  TOPIC ANALYSIS")
    print("-" * 80)

    topic_lengths = [len(t['headline'].split()) for t in topics]
    context_lengths = [len(t['context'].split()) for t in topics]

    print(f"Topic Headlines: Mean={statistics.mean(topic_lengths):.1f} words, "
          f"Range=[{min(topic_lengths)}, {max(topic_lengths)}]")
    print(f"Contexts:        Mean={statistics.mean(context_lengths):.1f} words, "
          f"Range=[{min(context_lengths)}, {max(context_lengths)}]")

    # Most common topic words
    all_topic_words = []
    for topic in topics:
        words = re.findall(r'\b[a-zA-Z]{4,}\b', topic['headline'].lower())
        all_topic_words.extend(words)

    common_topic_words = Counter(all_topic_words).most_common(10)
    print(f"\nMost common words in topic headlines:")
    for word, count in common_topic_words:
        print(f"  {word:20s}: {count:3d}")

    # 5. GENERATION TIME
    if 'generation_time' in articles[0]:
        print(f"\n   GENERATION TIME (seconds)")
        print("-" * 80)

        times_by_bias = defaultdict(list)
        all_times = []

        for article in articles:
            time = article['generation_time']
            all_times.append(time)
            times_by_bias[article['bias_type']].append(time)

        print(f"Overall: Mean={statistics.mean(all_times):.2f}s, "
              f"Total={sum(all_times)/60:.1f} minutes")

        print("\nBy Bias Type:")
        for bias_type in sorted(times_by_bias.keys()):
            times = times_by_bias[bias_type]
            print(f"  {bias_type:20s}: Mean={statistics.mean(times):.2f}s, "
                  f"Median={statistics.median(times):.2f}s")

    # 6. VOCABULARY RICHNESS
    print(f"\n  VOCABULARY ANALYSIS")
    print("-" * 80)

    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at',
                  'to', 'for', 'of', 'with', 'by', 'from', 'as', 'is',
                  'was', 'are', 'be', 'been', 'have', 'has', 'had'}

    words_by_bias = defaultdict(list)
    for article in articles:
        words = re.findall(r'\b[a-z]{4,}\b', article['article'].lower())
        filtered_words = [w for w in words if w not in stop_words]
        words_by_bias[article['bias_type']].extend(filtered_words)

    print("Unique words and vocabulary richness by bias:")
    for bias_type in sorted(words_by_bias.keys()):
        words = words_by_bias[bias_type]
        unique = len(set(words))
        total = len(words)
        richness = unique / total if total > 0 else 0
        print(f"  {bias_type:20s}: {unique:5,} unique / {total:6,} total "
              f"(richness: {richness:.3f})")

    # Most common words per bias
    print("\nMost common words by bias (top 5):")
    for bias_type in sorted(words_by_bias.keys()):
        top_words = Counter(words_by_bias[bias_type]).most_common(5)
        words_str = ", ".join([f"{w}({c})" for w, c in top_words])
        print(f"  {bias_type:20s}: {words_str}")

    # 7. DATA QUALITY CHECKS
    print(f"\n  DATA QUALITY CHECKS")
    print("-" * 80)

    # Check for duplicate topics
    topic_counter = Counter([a['topic'] for a in articles])
    duplicate_topics = [t for t, c in topic_counter.items() if c > 4]

    if duplicate_topics:
        print(f"   {len(duplicate_topics)} topics used more than 4 times")
        for topic in duplicate_topics[:3]:
            count = topic_counter[topic]
            print(f"   - {topic[:60]}... (used {count} times)")
    else:
        print("  No excessive topic reuse detected")

    # Check for very similar articles
    article_starts = defaultdict(list)
    for article in articles:
        start = article['article'][:100]
        article_starts[article['topic']].append(start)

    duplicate_articles = [(t, a) for t, a in article_starts.items()
                         if len(set(a)) < len(a)]

    if duplicate_articles:
        print(f"   {len(duplicate_articles)} topics with potentially duplicate articles")
    else:
        print(". No duplicate articles detected")

    # Check topic-article consistency
    print("\nVerifying topic-article consistency...")
    topic_set = {(t['headline'], t['context']) for t in topics}

    mismatches = []
    for article in articles:
        if (article['topic'], article['context']) not in topic_set:
            mismatches.append(article['topic'])

    if mismatches:
        print(f"   {len(mismatches)} articles don't match original topics")
        for topic in mismatches[:3]:
            print(f"   - {topic[:60]}...")
    else:
        print("  All articles match their source topics")

    print("\n" + "=" * 80)
    print("Analysis complete!")
    print("=" * 80)

compute_and_print_statistics(articles, topics_for_articles)

SUMMARY STATISTICS - BIAS POC DATA GENERATION

 OVERALL COUNTS
--------------------------------------------------------------------------------
Total Articles:       1,859
Unique Topics:        1,859
Unique Contexts:      1,859

 BIAS DISTRIBUTION
--------------------------------------------------------------------------------
center_left         :  465 articles (25.01%)
center_right        :  464 articles (24.96%)
strong_left         :  465 articles (25.01%)
strong_right        :  465 articles (25.01%)
  Distribution is well-balanced (within 1% difference)

  ARTICLE LENGTH (word count)
--------------------------------------------------------------------------------
Overall: Mean=266.1, Median=265.0, Std=30.5, Range=[181, 386]

By Bias Type:
  center_left         : Mean=261.6, Median=261.0, Std=27.1
  center_right        : Mean=257.2, Median=256.5, Std=27.6
  strong_left         : Mean=284.6, Median=285.0, Std=32.5
  strong_right        : Mean=260.9, Median=261.0, Std=26.7

   251 art

If the sanity check above reveals equal distribution of biases and no duplications, the topics & articles have been generated and stored successfully.

These articles, accessible at [kcorra716/bias-poc-data](https://huggingface.co/datasets/kcorra716/bias-poc-data), will be used to fine-tune both a baseline and a MST (monitor labeled) model. The models will be evaluated on their respective abilities to produce articles from the validation and test topics that are largely unbiased -- unlike the training articles we have generated here.